In [ ]:
# check available CUDA devices
import torch
devices = []
if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            device_name = torch.cuda.get_device_name(i)
            devices.append({
                "type": "CUDA",
                "available": True,
                "name": device_name,
                "index": i
            })
else:
    devices.append({"type": "CUDA", "available": False, "name": "N/A"})
devices

In [ ]:
# change folder into the root of the ASR project
import os

if not os.path.isdir("Configs"):
    %cd ../

!pwd

In [ ]:
# import packages, define common functions
import torch
import yaml
import os
from models import ASRCNN
from meldataset import build_dataloader
import torch.nn.functional as F
import pandas as pd
import os
import os.path as osp
from text_utils import TextCleaner
import itertools
from jiwer import wer
import re
from token_map import build_token_map_from_data

def cfg_get_nested(cfg: dict, path, default=None, sep='.'):
    """Get a nested value from a dict using a list of keys or a dot-separated string."""
    if isinstance(path, str):
        keys = path.split(sep) if path else []
    else:
        keys = path

    cur = cfg
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur

def length_to_mask(lengths):
    mask = torch.arange(lengths.max()).unsqueeze(0).expand(lengths.shape[0], -1).type_as(lengths)
    mask = torch.gt(mask + 1, lengths.unsqueeze(1))
    return mask

def resolve_ssl_frontend_config(config):
    ssl_cfg = cfg_get_nested(config, 'ssl_frontend', {}) or {}
    if not isinstance(ssl_cfg, dict):
        return {'enabled': False}
    resolved = dict(ssl_cfg)
    if not resolved.get('enabled', False):
        return {'enabled': False}

    default_models = {
        'wav2vec2': 'facebook/wav2vec2-base',
        'hubert': 'facebook/hubert-base-ls960',
        'wavlm': 'microsoft/wavlm-base',
        'xlsr': 'facebook/wav2vec2-large-xlsr-53',
    }
    selected_architecture = resolved.get('architecture')
    selected_model_name = resolved.get('pretrained_model_name')

    for candidate in ('wav2vec2', 'hubert', 'wavlm', 'xlsr'):
        candidate_cfg = resolved.get(candidate)
        if isinstance(candidate_cfg, dict) and candidate_cfg.get('enabled'):
            selected_architecture = candidate
            candidate_model = candidate_cfg.get('pretrained_model_name') or candidate_cfg.get('model_name')
            if candidate_model:
                selected_model_name = candidate_model
            break

    if selected_architecture is None:
        selected_architecture = 'wav2vec2'
    if not selected_model_name:
        selected_model_name = default_models.get(selected_architecture)

    if not selected_model_name:
        return {'enabled': False}

    resolved['architecture'] = selected_architecture
    resolved['pretrained_model_name'] = selected_model_name
    resolved['enabled'] = True
    return resolved

def get_preprocess_sample_rate(config):
    if 'preprocess_params' in config:
        key = 'preprocess_params'
    elif 'preprocess_parasm' in config:
        key = 'preprocess_parasm'
    else:
        key = None
    if key is None:
        return 24000
    return cfg_get_nested(config, f"{key}.sr", 24000)

def unpack_ctc_outputs(model_output):
    if isinstance(model_output, dict):
        logits = (
            model_output.get('logits_ctc')
            or model_output.get('ctc_logit')
            or model_output.get('logits')
            or model_output.get('ctc')
        )
        encoder_lengths = model_output.get('encoder_lengths')
    elif isinstance(model_output, (tuple, list)):
        logits = model_output[0] if len(model_output) > 0 else None
        encoder_lengths = None
        for item in model_output[1:]:
            if torch.is_tensor(item) and item.dim() == 1:
                encoder_lengths = item
                break
    else:
        logits = model_output
        encoder_lengths = None
    if logits is None:
        raise ValueError('Could not extract CTC logits from model output.')
    return logits, encoder_lengths

def prepare_batch_for_model(batch, use_ssl_frontend, device):
    if use_ssl_frontend:
        texts, text_lens, mels, mel_lens, wave_tensor, wave_lengths = batch
        features = wave_tensor.to(device)
        feature_lengths = wave_lengths.to(device)
        mel_inputs = mels
        mel_lengths = mel_lens
    else:
        texts, text_lens, mels, mel_lens = batch
        features = mels.to(device)
        feature_lengths = mel_lens.to(device)
        mel_inputs = mels
        mel_lengths = mel_lens
    return {
        'texts': texts,
        'text_lengths': text_lens,
        'mel_inputs': mel_inputs,
        'mel_lengths': mel_lengths,
        'features': features,
        'feature_lengths': feature_lengths,
        'feature_lengths_cpu': feature_lengths.detach().to(torch.long).cpu(),
    }

def load_ASR_models(ASR_MODEL_PATH, ASR_MODEL_CONFIG, device='cpu'):
    with open(ASR_MODEL_CONFIG) as f:
        config = yaml.safe_load(f)

    token_src = config.get('phoneme_maps_path')
    if not token_src:
        token_map = build_token_map_from_data(
            config.get('train_data'),
            config.get('val_data'),
            config.get('ood_data'),
            apply_asr_tokenizer=True,
        )
    elif isinstance(token_src, dict):
        token_map = token_src
    else:
        csv = pd.read_csv(token_src, header=None).values
        token_map = {word: index for word, index in csv}

    model_params = cfg_get_nested(config, 'model_params', {
        'input_dim': 80,
        'hidden_dim': 256,
        'n_token': len(token_map),
        'token_embedding_dim': 512,
        'n_layers': 5,
        'location_kernel_size': 31
    })

    if 'n_token' not in model_params:
        model_params['n_token'] = len(token_map)

    ssl_frontend_config = resolve_ssl_frontend_config(config)
    model = ASRCNN(**model_params, ssl_frontend_config=ssl_frontend_config)

    checkpoint = torch.load(ASR_MODEL_PATH, map_location=device, weights_only=False)
    params = checkpoint.get('model', checkpoint)
    try:
        model.load_state_dict(params)
    except Exception:
        new_state_dict = {k.replace('module.', ''): v for k, v in params.items()}
        model.load_state_dict(new_state_dict)

    model.to(device)
    model.eval()

    return model, config, token_map, ssl_frontend_config


In [ ]:
checkpoint_dir = "Checkpoint"

files = [f for f in os.listdir(checkpoint_dir + "/") if f.startswith('epoch_') and f.endswith('.pth')]
if not files:
    raise FileNotFoundError(f"No checkpoints found in {checkpoint_dir}.")
sorted_files = sorted(files, key=lambda x: int(x.split('_')[-1].split('.')[0]))

model_path = checkpoint_dir + "/" + sorted_files[-1]
# model_path = "Checkpoint/epoch_00080.pth"

config_path = 'Checkpoint/config.yml'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, config, token_map, ssl_frontend_config = load_ASR_models(model_path, config_path, device=device)
SSL_FRONTEND_ENABLED = bool(ssl_frontend_config.get('enabled', False))
INPUT_SAMPLE_RATE = get_preprocess_sample_rate(config)
TOKEN_MAP = token_map

phoneme_map_source = config.get('phoneme_maps_path')
if isinstance(phoneme_map_source, str):
    phoneme_map_display = phoneme_map_source
else:
    phoneme_map_display = 'built from dataset'

phoneme_map = token_map if not isinstance(phoneme_map_source, dict) else phoneme_map_source

test_csv_path = config['val_data']

def _dict_desc(obj):
    return obj if isinstance(obj, str) else 'built from dataset'

print("model --> " + model_path)
print("config --> " + config_path)
print("dictionary --> " + phoneme_map_display)
print("test: --> " + test_csv_path)
if SSL_FRONTEND_ENABLED:
    arch = ssl_frontend_config.get('architecture', 'unknown')
    pretrained_name = ssl_frontend_config.get('pretrained_model_name', 'unknown')
    print(f"SSL frontend enabled: {arch} ({pretrained_name})")
else:
    print("SSL frontend disabled; using mel-spectrogram features.")

model.eval()
print("All OK ✓")


In [ ]:
text_cleaner = TextCleaner(phoneme_map)
device = next(model.parameters()).device

with open(test_csv_path, "r", encoding="utf-8") as f:
    test_lines = f.readlines()

dataset_params = {
        'dict_path': cfg_get_nested(config, 'phoneme_maps_path', 'Data/word_index_dict.txt'),
        'sr': cfg_get_nested(config, 'preprocess_params.sr', cfg_get_nested(config, 'preprocess_parasm.sr', 24000)),
        'spect_params': cfg_get_nested( config, 'preprocess_params.spect_params', {
            'n_fft': 1024,
            'win_length': 1024,
            'hop_length': 300
        }),
        'mel_params': cfg_get_nested( config, 'preprocess_params.mel_params', { 'n_mels': 80 })
    }

collate_config = {"return_wave": SSL_FRONTEND_ENABLED}

test_loader = build_dataloader(
    path_list=[l[:-1].split('|') for l in test_lines],
    validation=True,
    batch_size=1,
    num_workers=2,
    device=str(device),
    collate_config=collate_config,
    dataset_config=dataset_params
)

wlist = phoneme_map
index2phoneme = {v: k for k, v in wlist.items()}

predictions = []
references = []
cleartexts = []

model.eval()
log_interval = 10
total = len(test_lines)
cntr = 0
maxtestsize = 1

with torch.no_grad():
    for batch in test_loader:
        cleartexts.append(test_lines[cntr])

        batch_data = prepare_batch_for_model(batch, SSL_FRONTEND_ENABLED, device)
        texts = batch_data['texts']
        input_lengths = batch_data['text_lengths']
        features = batch_data['features']
        feature_lengths = batch_data['feature_lengths']
        feature_lengths_cpu = batch_data['feature_lengths_cpu']

        model_kwargs = {'input_lengths': feature_lengths}
        if SSL_FRONTEND_ENABLED:
            model_kwargs['sampling_rate'] = INPUT_SAMPLE_RATE

        outputs = model(features, **model_kwargs)
        logits, encoder_lengths = unpack_ctc_outputs(outputs)
        predicted_ids = torch.argmax(logits, dim=-1)

        if encoder_lengths is None:
            encoder_lengths_cpu = feature_lengths_cpu
        else:
            encoder_lengths_cpu = encoder_lengths.detach().to(torch.long).cpu()

        texts_cpu = texts.detach().cpu()
        input_lengths_cpu = input_lengths.detach().to(torch.long).cpu()
        predicted_ids_cpu = predicted_ids.detach().cpu()

        seq_len = int(encoder_lengths_cpu[0].item()) if encoder_lengths_cpu.numel() > 0 else predicted_ids_cpu.size(1)
        seq_len = max(0, min(seq_len, predicted_ids_cpu.size(1)))
        predicted_tokens = predicted_ids_cpu[0, :seq_len]
        predicted_text = ''.join([text_cleaner.inverse_mapping.get(phoneme.item(), '<unk>') for phoneme in predicted_tokens])

        expected_text = test_lines[cntr].strip().split('|')[1]
        print(f"Batch {cntr} - Expected text: {expected_text}")
        print(f"Batch {cntr} - Predicted text: {predicted_text}")

        for i in range(predicted_ids_cpu.size(0)):
            cur_seq_len = int(encoder_lengths_cpu[i].item()) if encoder_lengths_cpu.numel() > i else predicted_ids_cpu.size(1)
            cur_seq_len = max(0, min(cur_seq_len, predicted_ids_cpu.size(1)))
            pred_seq = predicted_ids_cpu[i][:cur_seq_len]
            ref_seq = texts_cpu[i][:int(input_lengths_cpu[i].item())]

            pred_phonemes = [index2phoneme.get(p.item(), '') for p in pred_seq]
            ref_phonemes = [index2phoneme.get(r.item(), '') for r in ref_seq]

            predictions.append(pred_phonemes)
            references.append(ref_phonemes)

        cntr += 1
        if (cntr) % log_interval == 0:
            print(f"{cntr} of {total} sentences tested")

        if maxtestsize > 0 and cntr >= maxtestsize:
            print(f"early stop reached at {maxtestsize} sentences")
            break

print("Done - all sentences tested.")


In [ ]:
# Clean extra quotes and join tokens into space-separated strings
references_cleaned = [' '.join(token.strip('"') for token in seq) for seq in references]
predictions_cleaned = [' '.join(token.strip('"') for token in seq) for seq in predictions]

# Now use jiwer
per = wer(references_cleaned, predictions_cleaned)
print(f'Phoneme Error Rate: {per:.4f}')

In [ ]:
###########################################
# Find the best AUX model (with best PER) #
###########################################

files = [f for f in os.listdir(checkpoint_dir + "/") if f.startswith('epoch_') and f.endswith('.pth')]
sorted_files = sorted(files, key=lambda x: int(x.split('_')[-1].split('.')[0]))

model_path = checkpoint_dir + "/" + sorted_files[-1]
# model_path = "Checkpoint/epoch_00040.pth"
# config_path = "Configs/config.yml"

phoneme_map_source = config.get('phoneme_maps_path')
if not phoneme_map_source:
    phoneme_map_display = 'built from dataset'
elif isinstance(phoneme_map_source, dict):
    phoneme_map_display = 'built from dataset'
else:
    phoneme_map_display = phoneme_map_source

print("config --> " + config_path)
print("dictionary --> " + phoneme_map_display)
print("test --> " + test_csv_path)

text_cleaner = TextCleaner(phoneme_map)
device = next(model.parameters()).device

with open(test_csv_path, "r", encoding="utf-8") as f:
    test_lines = f.readlines()

maxtestsize = 25
best_model = ""
best_model_per = 100.0

model_cntr = 1
total_files = len(sorted_files)
results = []

for aux_model_file in sorted_files:
    model_path = checkpoint_dir + "/" + aux_model_file
    current_model, _, current_token_map, current_ssl_config = load_ASR_models(model_path, config_path, device=device)
    current_model.eval()

    current_ssl_enabled = bool(current_ssl_config.get('enabled', False))
    current_collate = {"return_wave": current_ssl_enabled}
    current_loader = build_dataloader(
        path_list=[l[:-1].split('|') for l in test_lines],
        validation=True,
        batch_size=1,
        num_workers=2,
        device=str(device),
        collate_config=current_collate,
        dataset_config=dataset_params
    )

    print(f"[{model_cntr}/{total_files}] Now evaluating AUX model: {aux_model_file}")
    model_cntr += 1

    predictions = []
    references = []
    first_ref = ""
    first_pred = ""

    current_index2phoneme = {v: k for k, v in current_token_map.items()}

    with torch.no_grad():
        for sample_idx, batch in enumerate(current_loader):
            if maxtestsize > 0 and sample_idx >= maxtestsize:
                break

            batch_data = prepare_batch_for_model(batch, current_ssl_enabled, device)
            features = batch_data['features']
            feature_lengths = batch_data['feature_lengths']
            feature_lengths_cpu = batch_data['feature_lengths_cpu']
            texts = batch_data['texts']
            text_lengths = batch_data['text_lengths']

            model_kwargs = {'input_lengths': feature_lengths}
            if current_ssl_enabled:
                model_kwargs['sampling_rate'] = INPUT_SAMPLE_RATE

            outputs = current_model(features, **model_kwargs)
            logits, encoder_lengths = unpack_ctc_outputs(outputs)
            predicted_ids = torch.argmax(logits, dim=-1)

            if encoder_lengths is None:
                encoder_lengths_cpu = feature_lengths_cpu
            else:
                encoder_lengths_cpu = encoder_lengths.detach().to(torch.long).cpu()

            predicted_ids_cpu = predicted_ids.detach().cpu()
            texts_cpu = texts.detach().cpu()
            text_lengths_cpu = text_lengths.detach().to(torch.long).cpu()

            for i in range(predicted_ids_cpu.size(0)):
                cur_seq_len = int(encoder_lengths_cpu[i].item()) if encoder_lengths_cpu.numel() > i else predicted_ids_cpu.size(1)
                cur_seq_len = max(0, min(cur_seq_len, predicted_ids_cpu.size(1)))
                pred_seq = predicted_ids_cpu[i][:cur_seq_len]
                ref_seq = texts_cpu[i][:int(text_lengths_cpu[i].item())]

                pred_phonemes = [current_index2phoneme.get(p.item(), '') for p in pred_seq]
                ref_phonemes = [current_index2phoneme.get(r.item(), '') for r in ref_seq]

                predictions.append(pred_phonemes)
                references.append(ref_phonemes)

                if first_ref == "":
                    first_ref = ' '.join(ref_phonemes)
                    first_pred = ' '.join(pred_phonemes)

    references_cleaned = [' '.join(seq) for seq in references]
    predictions_cleaned = [' '.join(seq) for seq in predictions]
    per = wer(references_cleaned, predictions_cleaned)

    print(f'Phoneme Error Rate: {per:.4f} {"✓" if per < best_model_per else "✗"}')
    if per < best_model_per:
        best_model_per = per
        best_model = aux_model_file

    results.append({
        'model': aux_model_file,
        'per': per,
        'first_ref': first_ref,
        'first_pred': first_pred
    })

results_sorted = sorted(results, key=lambda x: x['per'])

print("===================")
print("PERFORMANCE SUMMARY")
print("===================")
for res in results_sorted:
    print(f"Model: {res['model']}")
    print(f"PER: {res['per']:.4f}")
    print(f"Reference: {res['first_ref']}")
    print(f"Prediction: {res['first_pred']}")
    print("------------------------")

best = results_sorted[0]
print(f"✅ Best model: {best['model']} with PER = {best['per']:.4f}")
